# 02 - Preprocessing & Feature Engineering

Demonstrates how raw LAPD CSV is cleaned and what feature matrices are produced.

In [1]:
# bootstrap: make src importable
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import data_loader, preprocessing, feature_engineering
from src.utils import memory_mb


## Load a sample of the raw CSV

In [2]:
raw = data_loader.load_raw(nrows=200_000, use_cache=False)
print(f'memory: {memory_mb(raw):.1f} MB  shape: {raw.shape}')

02:07:27 | INFO    | src.data_loader | [start] read_csv Crime_Data_from_2020_to_Present.csv (nrows=200000)


02:07:29 | INFO    | src.data_loader | [done ] read_csv Crime_Data_from_2020_to_Present.csv (nrows=200000) in 1.83s


02:07:29 | INFO    | src.data_loader | loaded shape=(200000, 28)


memory: 70.8 MB  shape: (200000, 28)


## Clean & derive targets

In [3]:
clean = preprocessing.clean(raw)
print('new columns added:', sorted(set(clean.columns) - set(raw.rename(columns={c: c for c in raw.columns}).columns)))
clean[['date_occurred','hour','dow','is_weekend','is_violent','is_arrest','status_group','part_of_day']].head()

02:07:29 | INFO    | src.preprocessing | [start] preprocessing.clean


02:07:29 | INFO    | src.preprocessing | dropped 0 rows with missing date_occurred


02:07:29 | INFO    | src.preprocessing | [done ] preprocessing.clean in 0.32s


new columns added: ['address', 'area_id', 'area_name', 'crime_code', 'crime_code_1', 'crime_code_2', 'crime_code_3', 'crime_code_4', 'crime_desc', 'cross_street', 'date_occurred', 'date_reported', 'dow', 'dr_no', 'has_coords', 'hour', 'is_arrest', 'is_violent', 'is_weekend', 'lat', 'lon', 'mo_codes', 'month', 'part_class', 'part_of_day', 'premise_code', 'premise_desc', 'report_district', 'report_lag_days', 'status_code', 'status_desc', 'status_group', 'time_occurred', 'victim_age', 'victim_descent', 'victim_sex', 'weapon_code', 'weapon_desc', 'weapon_used', 'week', 'year']


,date_occurred,hour,dow,is_weekend,is_violent,is_arrest,status_group,part_of_day
0,2020-01-01,8,2,False,0,0,Investigation Continued,Morning
1,2020-01-01,8,2,False,0,0,Investigation Continued,Morning
2,2020-01-01,12,2,False,1,0,Investigation Continued,Afternoon
3,2020-01-01,9,2,False,0,0,Investigation Continued,Morning
4,2020-01-01,16,2,False,1,0,Adult Other,Afternoon


## Build the incident-level feature matrix

In [4]:
inc = feature_engineering.build_incident_features(clean)
print('features shape:', inc.shape)
inc.head(3)

features shape: (200000, 27)


,area_id,area_name,hour,dow,month,year,is_weekend,part_of_day,victim_age,victim_sex,...,crime_desc,part_class,is_violent,is_arrest,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos
0,19,Mission,8,2,1,2020,False,Morning,15.0,F,...,"LETTERS, LEWD - TELEPHONE CALLS, LEWD",2,0,0,8.660254e-01,-0.5,0.974928,-0.222521,0.5,0.866025
1,13,Newton,8,2,1,2020,False,Morning,83.0,F,...,"THEFT-GRAND ($950.01 & OVER)EXCPT,GUNS,FOWL,LI...",1,0,0,8.660254e-01,-0.5,0.974928,-0.222521,0.5,0.866025
2,18,Southeast,12,2,1,2020,False,Afternoon,50.0,M,...,ROBBERY,1,1,0,1.224647e-16,-1.0,0.974928,-0.222521,0.5,0.866025


## Build daily and weekly area panels

In [5]:
daily = feature_engineering.build_daily_area_panel(clean)
print('daily panel:', daily.shape)
daily.head()

02:07:29 | INFO    | src.feature_engineering | [start] build_daily_area_panel


02:07:29 | INFO    | src.feature_engineering | [done ] build_daily_area_panel in 0.05s


daily panel: (15267, 8)


,date,area_id,area_name,crimes,violent,arrests,violent_ratio,arrest_ratio
0,2020-01-01,1,Central,59.0,12.0,5.0,0.203390,0.084746
1,2020-01-01,2,Rampart,68.0,11.0,7.0,0.161765,0.102941
2,2020-01-01,3,Southwest,51.0,9.0,5.0,0.176471,0.098039
3,2020-01-01,4,Hollenbeck,41.0,9.0,5.0,0.219512,0.121951
4,2020-01-01,5,Harbor,47.0,10.0,9.0,0.212766,0.191489


In [6]:
weekly = feature_engineering.build_weekly_area_panel(daily)
print('weekly panel:', weekly.shape)
weekly.head()

02:07:29 | INFO    | src.feature_engineering | [start] build_weekly_area_panel


02:07:29 | INFO    | src.feature_engineering | [done ] build_weekly_area_panel in 0.01s


weekly panel: (2205, 8)


,date,area_id,area_name,crimes,violent,arrests,violent_ratio,arrest_ratio
0,2019-12-30,1,Central,207.0,33.0,15.0,0.159420,0.072464
1,2019-12-30,2,Rampart,157.0,29.0,20.0,0.184713,0.127389
2,2019-12-30,3,Southwest,182.0,32.0,10.0,0.175824,0.054945
3,2019-12-30,4,Hollenbeck,138.0,24.0,15.0,0.173913,0.108696
4,2019-12-30,5,Harbor,151.0,34.0,23.0,0.225166,0.152318


## Trim incomplete reporting tail

In [7]:
trimmed = feature_engineering.trim_incomplete_tail(daily)
print('before:', daily['date'].max(), '  after:', trimmed['date'].max())

before: 2021-12-27 00:00:00   after: 2021-12-27 00:00:00
